In [1]:
#creation of a file with korean restaraunts on google maps
import asyncio
import re
import nest_asyncio
import pandas as pd
from playwright.async_api import async_playwright

# patch google colab's active asyncio event loop
nest_asyncio.apply()


async def scrape_google_maps_korean_places():
    async with async_playwright() as p:
        # launch headless browser with required linux container flags
        browser = await p.chromium.launch(
            headless=True, args=["--no-sandbox", "--disable-setuid-sandbox"]
        )
        context = await browser.new_context(locale="en-US")
        page = await context.new_page()

        # search for korean restaurants in tashkent
        url = "https://www.google.com/maps/search/Korean+restaurant+in+Tashkent"
        await page.goto(url)
        await page.wait_for_timeout(3000)

        # scroll sidebar container to load more results dynamically
        sidebar_selector = 'div[role="feed"]'
        try:
            await page.wait_for_selector(sidebar_selector, timeout=5000)
            for _ in range(10):
                await page.eval_on_selector(
                    sidebar_selector, "e => e.scrollTop = e.scrollHeight"
                )
                await page.wait_for_timeout(2000)
        except Exception as e:
            print(f"Scroll notification: {e}")

        # target place links inside search results
        cards = await page.query_selector_all(
            "a.hfSNrf, a.hfHK1b, a[href*='/maps/place/']"
        )
        places = []
        seen_urls = set()

        for card in cards:
            href = await card.get_attribute("href")
            aria_label = await card.get_attribute("aria-label")

            if href and href not in seen_urls and "/maps/place/" in href:
                seen_urls.add(href)

                lat, lng = None, None

                # pattern 1: google maps place url params (!3d = latitude, !4d = longitude)
                coord_match = re.search(r"!3d(-?\d+\.\d+)!4d(-?\d+\.\d+)", href)
                if coord_match:
                    lat = float(coord_match.group(1))
                    lng = float(coord_match.group(2))
                else:
                    # pattern 2: standard @lat,lng format
                    at_match = re.search(r"@(-?\d+\.\d+),(-?\d+\.\d+)", href)
                    if at_match:
                        lat = float(at_match.group(1))
                        lng = float(at_match.group(2))

                # clean place name
                name = (
                    aria_label
                    if aria_label
                    else (
                        href.split("/maps/place/")[1]
                        .split("/")[0]
                        .replace("+", " ")
                    )
                )

                places.append(
                    {
                        "name": name,
                        "lat": lat,
                        "lon": lng,
                        "Google Maps Link": href,
                    }
                )

        await browser.close()

        # deduplicate results by name and coordinates
        df = pd.DataFrame(places)
        df = df.drop_duplicates(subset=["name", "lat", "lon"])

        # save to CSV with utf-8-sig encoding for clear character rendering in Excel
        output_filename = "tashkent_korean_google_scraped.csv"
        df.to_csv(output_filename, index=False, encoding="utf-8-sig")

        print(f"Done! Scraped {len(df)} places with Latitude and Longitude.")


# execute directly in google colab
await scrape_google_maps_korean_places()

ModuleNotFoundError: No module named 'playwright'

In [31]:
import folium
import pandas as pd
data = pd.read_csv('tashkent_korean_google_scraped.csv')

m = folium.Map(location=[41.3, 69.28], zoom_start=10)

for i, row in data.iterrows():
    folium.Marker(
        [row['lat'], row['lon']],
        popup=row['name']
    ).add_to(m)

m
